# 03. Градиентный бустинг и подбор параметров (этап 3)
LightGBM; параметры подбираются случайным поиском с оценкой по **TimeSeriesSplit** внутри train (обычный KFold не используется); ранняя остановка по AUC на validation; калибровка isotonic на validation.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

In [ ]:
r3 = P.ensure("step3")
print("движок:", r3["engine"], "| лучшая итерация:", r3["best_iteration"], "| признаков:", r3["n_features_model"])
r3["final_params"]

## Подбор по времени
Таблица поиска (топ-10 по среднему Gini на окнах TimeSeriesSplit). Итоговые параметры записаны в `config.yaml` (раздел `boosting.final_params`).

In [ ]:
pd.read_csv("artifacts/search_results.csv").head(10)

## Сравнение с базовой моделью — на одном и том же validation

In [ ]:
cmp = pd.read_csv("artifacts/comparison_valid.csv"); cmp

In [ ]:
print("95% ДИ Gini бустинга на validation:", [round(x, 3) for x in r3["valid_gini_ci"]])
print("Gini на validation на сырых признаках (без 15 новых):", round(r3["gini_valid_raw_features_only"], 3),
      "| со всеми признаками:", round(r3["valid_raw"]["gini"], 3))

## Калибровка
Isotonic-калибровка (`CalibratedClassifierCV`, модель заморожена, калибратор обучен на validation). Кривая ниже построена на validation, т.е. для isotonic она «в выборке»; честная проверка калибровки — на test в ноутбуке 04.

In [ ]:
P.show_fig("fig_calibration_valid.png")

**Вывод.** LightGBM (296 деревьев, 16 листьев, шаг 0,02, scale_pos_weight = 3) по TimeSeriesSplit на train даёт Gini 0,840 против 0,795 у скоринговой карты, но разброс по окнам около 0,09, так что выигрыш небольшой. На validation модели почти равны (Gini 0,969 у обеих; 13 банкротов). Калибровка isotonic снижает Brier с 0,0114 до 0,0083, но на validation это оценка «в выборке». Новые признаки на validation Gini не подняли (0,974 без них, 0,969 с ними), разница в пределах шума.